# Metajeux-05 : Le temoin d'impossibilite

**Navigation** : [<< Metajeux-04-Chemin-Minimal-Robinson-Goforth](Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) | [Version Python](Metajeux-05-Temoins-Impossibilite-Python.ipynb) | [Index](README.md) | [Metajeux-06-Chemin-Minimal-3x2 >>](Metajeux-06-Chemin-Minimal-3x2-Ordinal-Python.ipynb)

**Kernel** : .NET (C#)

GT-20 construit et verifie un chemin minimal entre deux jeux ; ce carnet attaque la question duale, le **temoin d'impossibilite** — prouver qu'aucun chemin plus court n'existe — en s'appuyant sur la **structure produit** des chambres : `d_chambre(G,H) = d_perm(row_G,row_H) + d_perm(col_G,col_H)`.

**Conventions reprises de GT-20** : `Canonique`, `SwapValeursAdjacentes`, `SwapJeu(jeu, cote, k)`, 24 ordres stricts sur 4 cases, 576 chambres, 6 voisins par jeu (3 swaps x {ligne, colonne}).

## 0. Configuration

In [1]:
// === Configuration : GameTheory 20b ===
using System;
using System.Collections.Generic;
using System.Linq;

// Chambre = paire d'ordres stricts sur 4 cases (ligne, colonne).
// Port .NET : un `record` au lieu d'un tuple Python (egalite par valeur, cle de dictionnaire).
public sealed record Chambre(string Row, string Col)
{
    public override string ToString() => $"({Row},{Col})";
}

// Canonique : relabele les valeurs distinctes par ordre croissant (1, 2, ...).
static string Canonique(IEnumerable<int> t)
{
    var vals = t.Distinct().OrderBy(v => v).ToList();
    return string.Concat(t.Select(v => (vals.IndexOf(v) + 1).ToString()));
}

// SwapValeursAdjacentes : echange les positions des valeurs k et k+1.
static string SwapValeursAdjacentes(string t, int k)
{
    var l = t.ToCharArray();
    int pk = Array.IndexOf(l, (char)('0' + k));
    int pk1 = Array.IndexOf(l, (char)('0' + (k + 1)));
    (l[pk], l[pk1]) = (l[pk1], l[pk]);
    return new string(l);
}

// SwapJeu : un swap sur un seul cote (ligne ou colonne).
static Chambre SwapJeu(Chambre jeu, string cote, int k)
{
    return cote == "ligne"
        ? new Chambre(SwapValeursAdjacentes(jeu.Row, k), jeu.Col)
        : new Chambre(jeu.Row, SwapValeursAdjacentes(jeu.Col, k));
}

// AdjChambre : 6 voisins par chambre (3 swaps x {ligne, colonne}).
static List<Chambre> AdjChambre(Chambre g)
{
    var voisins = new List<Chambre>();
    foreach (var cote in new[] { "ligne", "colonne" })
    {
        foreach (int k in new[] { 1, 2, 3 })
        {
            voisins.Add(SwapJeu(g, cote, k));
        }
    }
    return voisins;
}

// AdjPerm : les 3 voisins d'un ordre strict dans le permutaedre.
static List<string> AdjPerm(string t)
{
    var voisins = new List<string>();
    for (int k = 1; k <= 3; k++)
    {
        voisins.Add(SwapValeursAdjacentes(t, k));
    }
    return voisins;
}

// BfsPerm : distances depuis un ordre strict (par source).
static Dictionary<string, int> BfsPerm(string depart, Func<string, IEnumerable<string>> voisinsFn)
{
    var dist = new Dictionary<string, int> { [depart] = 0 };
    var q = new Queue<string>();
    q.Enqueue(depart);
    while (q.Count > 0)
    {
        var u = q.Dequeue();
        foreach (var v in voisinsFn(u))
        {
            if (!dist.ContainsKey(v))
            {
                dist[v] = dist[u] + 1;
                q.Enqueue(v);
            }
        }
    }
    return dist;
}

// BfsChambre : distances depuis une chambre (par source).
static Dictionary<Chambre, int> BfsChambre(Chambre depart, Func<Chambre, IEnumerable<Chambre>> voisinsFn)
{
    var dist = new Dictionary<Chambre, int> { [depart] = 0 };
    var q = new Queue<Chambre>();
    q.Enqueue(depart);
    while (q.Count > 0)
    {
        var u = q.Dequeue();
        foreach (var v in voisinsFn(u))
        {
            if (!dist.ContainsKey(v))
            {
                dist[v] = dist[u] + 1;
                q.Enqueue(v);
            }
        }
    }
    return dist;
}

// Les 24 ordres stricts sur 4 cases (port manuel de itertools.permutations).
static List<string> PermutationsStrictes()
{
    var res = new List<string>();
    void Backtrack(List<int> prefix)
    {
        if (prefix.Count == 4)
        {
            res.Add(Canonique(prefix));
            return;
        }
        foreach (int v in new[] { 1, 2, 3, 4 })
        {
            if (prefix.Contains(v))
            {
                continue;
            }
            prefix.Add(v);
            Backtrack(prefix);
            prefix.RemoveAt(prefix.Count - 1);
        }
    }
    Backtrack(new List<int>());
    return res.Distinct().OrderBy(s => s, StringComparer.Ordinal).ToList();
}

var stricts = PermutationsStrictes();
var chambres = (from r in stricts from c in stricts select new Chambre(r, c)).ToList();

Console.WriteLine("GameTheory 20b : temoin d'impossibilite certifie sur R-G (point 4)");
Console.WriteLine();
Console.WriteLine($"Permutaedre : {stricts.Count} ordres stricts, {AdjPerm(stricts[0]).Count} voisins / ordre");
Console.WriteLine($"Chambres : {chambres.Count} sommets, {AdjChambre(chambres[0]).Count} voisins / chambre");

GameTheory 20b : temoin d'impossibilite certifie sur R-G (point 4)


Permutaedre : 24 ordres stricts, 3 voisins / ordre


Chambres : 576 sommets, 6 voisins / chambre


## 1. La structure produit

Le graphe des chambres est un **produit cartesien** de deux copies du permutaedre `S_4` : chaque chambre `(row, col)` est une paire d'ordres stricts sur 4 cases, et les generateurs ne touchent qu'un **seul cote a la fois** (un swap sur la ligne laisse la colonne intacte, et reciproquement).

**Theoreme cle** : pour tous `G = (row_G, col_G)` et `H = (row_H, col_H)` dans `chambres`, la distance dans le produit est exactement la somme des distances dans chaque facteur : `d_chambre(G, H) = d_perm(row_G, row_H) + d_perm(col_G, col_H)`.

**Demonstration par double inegalite** :

- **Borne inferieure (par comptage des pas de chaque cote)** : soit un chemin de `G` a `H` de longueur `L` dans le produit. Chaque generateur ne touche qu'un seul cote, donc les `L` pas se repartissent en `n_row` pas cote ligne et `n_col` pas cote colonne, avec `L = n_row + n_col`. En ne gardant que les pas cote ligne, on obtient un chemin de `row_G` a `row_H` dans le permutaedre, de longueur `n_row` : donc `n_row >= d_perm(row_G, row_H)`, et symetriquement `n_col >= d_perm(col_G, col_H)`. En sommant : `L >= d_perm(row_G, row_H) + d_perm(col_G, col_H)`.

  *Pourquoi la projection seule ne suffit pas* : projeter le chemin **entier** sur le facteur ligne ne preserve pas la longueur — les pas cote colonne projettent sur du surplace. On en tire `d_perm(row_G, row_H) <= L`, et de meme `d_perm(col_G, col_H) <= L` : cela borne `L` par le **maximum** des deux distances, pas par leur **somme**. C'est le comptage `L = n_row + n_col`, et lui seul, qui donne l'additivite.

- **Borne superieure (par concatenation)** : reciproquement, un chemin optimal dans le facteur ligne (longueur `d_perm(row_G, row_H)`) et un chemin optimal dans le facteur colonne (longueur `d_perm(col_G, col_H)`) peuvent etre concatenes cote par cote pour produire un chemin de longueur `d_perm(row_G, row_H) + d_perm(col_G, col_H)` dans le produit (l'independance des cotes permet l'entrelacement arbitraire). Donc `d_chambre(G, H) <= d_perm(row_G, row_H) + d_perm(col_G, col_H)`.

- **Conclusion** : l'egalite stricte est prouvee pour **toute** paire `(G, H)`. Le produit cartesien preserve exactement la distance metrique — c'est la structure produit du graphe.

In [2]:
// === Section 1.1 : verification exhaustive ===
using System;
using System.Collections.Generic;
using System.Linq;

var IDENTITE = new Chambre("1234", "1234");

// Distances dans le permutaedre, par source : Dictionary<string, Dictionary<string, int>>.
var dist_perm = stricts.ToDictionary(o => o, o => BfsPerm(o, AdjPerm));

// Distances dans le produit des chambres, par source : Dictionary<Chambre, int>.
var dist_chambre = BfsChambre(IDENTITE, AdjChambre);

var ecarts = new List<(Chambre Chambre, int DChambre, int DCalc)>();
foreach (var ch in chambres)
{
    int d_ch = dist_chambre[ch];
    int d_calc = dist_perm[IDENTITE.Row][ch.Row] + dist_perm[IDENTITE.Col][ch.Col];
    if (d_ch != d_calc)
    {
        ecarts.Add((ch, d_ch, d_calc));
    }
}

int diam_perm = dist_perm.Values.Max(d => d.Values.Max());
int diam_chambres = dist_chambre.Values.Max();

Console.WriteLine($"Verification exhaustive sur {chambres.Count} chambres : {ecarts.Count} ecarts");
Console.WriteLine($"Structure produit : {(ecarts.Count == 0 ? "EXACTE" : "NON")}");
Console.WriteLine($"Diametre permutaedre : {diam_perm} (attendu 6) -> {(diam_perm == 6 ? "CONFORME" : "ECART")}");
Console.WriteLine($"Diametre chambres : {diam_chambres} (attendu 12) -> {(diam_chambres == 12 ? "CONFORME" : "ECART")}");

// Detail des ecarts eventuels (aucun attendu si la structure produit tient).
foreach (var e in ecarts.Take(10))
{
    Console.WriteLine($"  ECART : {e.Chambre} d_chambre={e.DChambre} d_row+d_col={e.DCalc}");
}

Console.WriteLine($"Structure produit : {chambres.Count - ecarts.Count}/{chambres.Count} chambres -> {(ecarts.Count == 0 ? "CONFORME" : $"ECART : {ecarts.Count} ecarts")}");
if (ecarts.Count != 0)
{
    throw new InvalidOperationException($"Structure produit invalide : {ecarts.Count} ecarts");
}
Console.WriteLine($"ASSERTION PASS : structure produit verifiee sur {chambres.Count}/{chambres.Count}");

Verification exhaustive sur 576 chambres : 0 ecarts


Structure produit : EXACTE


Diametre permutaedre : 6 (attendu 6) -> CONFORME


Diametre chambres : 12 (attendu 12) -> CONFORME


Structure produit : 576/576 chambres -> CONFORME


ASSERTION PASS : structure produit verifiee sur 576/576


## 2. Le certificat analytique

Pour un triplet `(G, H, k_max)`, on definit le verdict par comparaison :

- **`IMPOSSIBLE`** si `d_row + d_col > k_max` (la distance minimale theorique depasse la borne, donc aucun chemin de longueur `<= k_max` n'existe) ;
- **`POSSIBLE`** si `d_row + d_col <= k_max` (un chemin de cette longueur existe, par concatenation des chemins optimaux dans chaque facteur).

Le verdict `IMPOSSIBLE` porte un **certificat analytique** : les valeurs `(d_row, d_col, k_max)` qui le justifient. Le verdict `POSSIBLE` porte la meme garantie d'existence, mais ne fournit pas le chemin explicite — celui-ci est produit par le constructeur de GT-20 (point 1 du critere #12205 §5).

**Verification de la table 24x24** : le tableau `D` des distances dans le permutaedre doit etre symetrique et triangulaire pour que le certificat soit coherent.

In [3]:
// === Section 2.1 : precalcul 24x24 ===
using System;
using System.Collections.Generic;
using System.Linq;

// Table D des distances dans le permutaedre, indexee par la paire d'ordres.
var D = new Dictionary<(string, string), int>();
foreach (var o1 in stricts)
{
    foreach (var o2 in stricts)
    {
        D[(o1, o2)] = dist_perm[o1][o2];
    }
}

int non_sym = 0;
foreach (var o1 in stricts)
{
    foreach (var o2 in stricts)
    {
        if (D[(o1, o2)] != D[(o2, o1)])
        {
            non_sym++;
        }
    }
}
Console.WriteLine($"Symetrie 24x24 : {non_sym} asymetries");

int violations = 0;
foreach (var o1 in stricts)
{
    foreach (var o2 in stricts)
    {
        foreach (var o3 in stricts)
        {
            if (D[(o1, o3)] > D[(o1, o2)] + D[(o2, o3)])
            {
                violations++;
            }
        }
    }
}
int nb_triplets = stricts.Count * stricts.Count * stricts.Count;
Console.WriteLine($"Inegalite triangulaire : {violations} violations sur {nb_triplets} triplets");
Console.WriteLine($"Symetrie : {(non_sym == 0 ? "CONFORME (0 asymetrie)" : $"ECART : {non_sym} asymetries")} ; triangularite : {(violations == 0 ? "CONFORME (0 violation)" : $"ECART : {violations} violations")}");
if (violations != 0)
{
    throw new InvalidOperationException($"Table 24x24 non triangulaire : {violations} violations");
}
Console.WriteLine("ASSERTION PASS : table 24x24 symetrique et triangulaire");

Symetrie 24x24 : 0 asymetries


Inegalite triangulaire : 0 violations sur 13824 triplets


Symetrie : CONFORME (0 asymetrie) ; triangularite : CONFORME (0 violation)


ASSERTION PASS : table 24x24 symetrique et triangulaire


## 3. Le temoin en action

Trois cas fondateurs illustrent les regimes du certificat :

- **Cas A** : antipode `IDENTITE -> RENVERS`, `k_max=5` -> verdict `IMPOSSIBLE` (distance 12 > 5) ;
- **Cas B** : `IDENTITE` vers une chambre a `d_row=0, d_col=4`, `k_max=3` -> verdict `IMPOSSIBLE` (distance 4 > 3) ;
- **Cas C** : frontiere `d_row=0, d_col=1`, `k_max=1` -> verdict `POSSIBLE` (distance 1 <= 1).

**Note sur le Cas C** : pour `H_C = ((1, 2, 4, 3), (1, 2, 3, 4))`, la table `D` donne `d_row = 1` (la ligne `1243` est a un seul swap de `1234`) et `d_col = 0` (les colonnes sont identiques). C'est donc la ligne de preuve `1 + 0 = 1 <= 1` qui fait foi : la valeur `1` porte sur la **ligne**, pas sur la colonne.

L'exemple ci-dessous montre comment le verdict est obtenu a partir de la table `D`. Le retour est un enregistrement `Certificat` (champs `Verdict`, `DRow`, `DCol`, `DMin`, `KMax`, `Preuve`) ou `Preuve` est la chaine qui materialise l'inegalite.

In [4]:
// === Section 3.1 : certifier_impossibilite ===
using System;
using System.Collections.Generic;

// Certificat analytique : porte le verdict et les valeurs qui le justifient.
public sealed record Certificat(string Verdict, int DRow, int DCol, int DMin, int KMax, string Preuve);

// CertifierImpossibilite : verdict analytique en O(1) a partir de la table D.
static Certificat CertifierImpossibilite(Chambre G, Chambre H, int kMax, Dictionary<(string, string), int> table)
{
    int d_row = table[(G.Row, H.Row)];
    int d_col = table[(G.Col, H.Col)];
    int d_min = d_row + d_col;
    if (d_min > kMax)
    {
        return new Certificat("IMPOSSIBLE", d_row, d_col, d_min, kMax,
            $"d_perm(row) + d_perm(col) = {d_row} + {d_col} = {d_min} > {kMax}");
    }
    else
    {
        return new Certificat("POSSIBLE", d_row, d_col, d_min, kMax,
            $"d_perm(row) + d_perm(col) = {d_row} + {d_col} = {d_min} <= {kMax}");
    }
}

var RENVERS = new Chambre("4321", "4321");
var cas_A = CertifierImpossibilite(IDENTITE, RENVERS, 5, D);
Console.WriteLine($"CAS A (antipode, k_max=5) : {cas_A.Verdict} - {cas_A.Preuve}");
Console.WriteLine($"  d_row={cas_A.DRow} (attendu 6), d_col={cas_A.DCol} (attendu 6), d_min={cas_A.DMin} (attendu 12) -> {(cas_A.DRow == 6 && cas_A.DCol == 6 && cas_A.DMin == 12 ? "CONFORME" : "ECART")}");
Console.WriteLine($"  verdict attendu IMPOSSIBLE -> {(cas_A.Verdict == "IMPOSSIBLE" ? "CONFORME" : "ECART")}");

var H_B = new Chambre("1234", "3412");
var cas_B = CertifierImpossibilite(IDENTITE, H_B, 3, D);
Console.WriteLine($"CAS B (row identique, k_max=3) : {cas_B.Verdict} - {cas_B.Preuve}");
Console.WriteLine($"  d_row={cas_B.DRow} (attendu 0), d_col={cas_B.DCol} (attendu 4), d_min={cas_B.DMin} (attendu 4) -> {(cas_B.DRow == 0 && cas_B.DCol == 4 && cas_B.DMin == 4 ? "CONFORME" : "ECART")}");
Console.WriteLine($"  verdict attendu IMPOSSIBLE -> {(cas_B.Verdict == "IMPOSSIBLE" ? "CONFORME" : "ECART")}");

// H_C = ((1, 2, 4, 3), (1, 2, 3, 4)) : la ligne est a 1 swap de l'identite (d_row = 1),
// les colonnes sont identiques (d_col = 0) -- conforme a la ligne de preuve "1 + 0 = 1 <= 1".
var H_C = new Chambre("1243", "1234");
var cas_C = CertifierImpossibilite(IDENTITE, H_C, 1, D);
Console.WriteLine($"CAS C (frontiere, k_max=1) : {cas_C.Verdict} - {cas_C.Preuve}");
Console.WriteLine($"  d_row={cas_C.DRow} (attendu 1), d_col={cas_C.DCol} (attendu 0), d_min={cas_C.DMin} (attendu 1) -> {(cas_C.DRow == 1 && cas_C.DCol == 0 && cas_C.DMin == 1 ? "CONFORME" : "ECART")}");
Console.WriteLine($"  verdict attendu POSSIBLE -> {(cas_C.Verdict == "POSSIBLE" ? "CONFORME" : "ECART")}");

CAS A (antipode, k_max=5) : IMPOSSIBLE - d_perm(row) + d_perm(col) = 6 + 6 = 12 > 5


  d_row=6 (attendu 6), d_col=6 (attendu 6), d_min=12 (attendu 12) -> CONFORME


  verdict attendu IMPOSSIBLE -> CONFORME


CAS B (row identique, k_max=3) : IMPOSSIBLE - d_perm(row) + d_perm(col) = 0 + 4 = 4 > 3


  d_row=0 (attendu 0), d_col=4 (attendu 4), d_min=4 (attendu 4) -> CONFORME


  verdict attendu IMPOSSIBLE -> CONFORME


CAS C (frontiere, k_max=1) : POSSIBLE - d_perm(row) + d_perm(col) = 1 + 0 = 1 <= 1


  d_row=1 (attendu 1), d_col=0 (attendu 0), d_min=1 (attendu 1) -> CONFORME


  verdict attendu POSSIBLE -> CONFORME


## 4. Verification BFS tronque

Pour chaque cas IMPOSSIBLE, on verifie par BFS tronque que `H` n'est **jamais** dans les sommets a distance `<= k_max` de `G`. Le BFS tronque collecte tous les sommats atteignables en au plus `k_max` pas, sans reconstruire le graphe entier — c'est un outil de verification operationnelle, distinct du certificat analytique et utile comme contre-controle.

En .NET, `BfsTronque(depart, voisinsFn, profondeurMax)` renvoie l'ensemble (`HashSet<Chambre>`) des sommets atteignables en au plus `profondeurMax` pas.

In [5]:
// === Section 4.1 : BFS tronque ===
using System;
using System.Collections.Generic;
using System.Linq;

// BfsTronque : sommets atteignables en au plus profondeurMax pas (sans reconstruire le graphe entier).
static HashSet<Chambre> BfsTronque(Chambre depart, Func<Chambre, IEnumerable<Chambre>> voisinsFn, int profondeurMax)
{
    var atteints = new HashSet<Chambre> { depart };
    var frontiere = new HashSet<Chambre> { depart };
    for (int d = 0; d < profondeurMax; d++)
    {
        var nouvelleFrontiere = new HashSet<Chambre>();
        foreach (var u in frontiere)
        {
            foreach (var v in voisinsFn(u))
            {
                if (!atteints.Contains(v))
                {
                    atteints.Add(v);
                    nouvelleFrontiere.Add(v);
                }
            }
        }
        frontiere = nouvelleFrontiere;
        if (frontiere.Count == 0)
        {
            break;
        }
    }
    return atteints;
}

// Taille de la boule attendue d'apres la table D (croisement analytique / operationnel).
static int TailleBoulePrevue(Dictionary<(string, string), int> table, Chambre G, int kMax, List<Chambre> toutes)
{
    return toutes.Count(c => table[(G.Row, c.Row)] + table[(G.Col, c.Col)] <= kMax);
}

var atte_A = BfsTronque(IDENTITE, AdjChambre, 5);
bool verif_A = !atte_A.Contains(RENVERS);
int prevu_A = TailleBoulePrevue(D, IDENTITE, 5, chambres);
Console.WriteLine($"CAS A : sommets a <= 5 pas = {atte_A.Count}, prevu par la table D = {prevu_A} -> {(atte_A.Count == prevu_A ? "CONFORME" : "ECART")}");
Console.WriteLine($"  RENVERS present ? {atte_A.Contains(RENVERS)} (attendu : False)");
Console.WriteLine($"  Temoin IMPOSSIBLE verifie : {verif_A}");

var atte_B = BfsTronque(IDENTITE, AdjChambre, 3);
bool verif_B = !atte_B.Contains(H_B);
int prevu_B = TailleBoulePrevue(D, IDENTITE, 3, chambres);
Console.WriteLine($"CAS B : sommets a <= 3 pas = {atte_B.Count}, prevu par la table D = {prevu_B} -> {(atte_B.Count == prevu_B ? "CONFORME" : "ECART")}");
Console.WriteLine($"  H_B present ? {atte_B.Contains(H_B)} (attendu : False)");
Console.WriteLine($"  Temoin IMPOSSIBLE verifie : {verif_B}");

var atte_C = BfsTronque(IDENTITE, AdjChambre, 1);
bool verif_C = atte_C.Contains(H_C);
int prevu_C = TailleBoulePrevue(D, IDENTITE, 1, chambres);
Console.WriteLine($"CAS C : sommets a <= 1 pas = {atte_C.Count}, prevu par la table D = {prevu_C} -> {(atte_C.Count == prevu_C ? "CONFORME" : "ECART")}");
Console.WriteLine($"  H_C present ? {atte_C.Contains(H_C)} (attendu : True)");
Console.WriteLine($"  Temoin POSSIBLE verifie : {verif_C}");

if (!(verif_A && verif_B && verif_C))
{
    throw new InvalidOperationException("Controle BFS tronque en echec sur au moins un cas");
}
Console.WriteLine("ASSERTION PASS : 3 cas (2 IMPOSSIBLE, 1 POSSIBLE) verifies");

CAS A : sommets a <= 5 pas = 235, prevu par la table D = 235 -> CONFORME


  RENVERS present ? False (attendu : False)


  Temoin IMPOSSIBLE verifie : True


CAS B : sommets a <= 3 pas = 68, prevu par la table D = 68 -> CONFORME


  H_B present ? False (attendu : False)


  Temoin IMPOSSIBLE verifie : True


CAS C : sommets a <= 1 pas = 7, prevu par la table D = 7 -> CONFORME


  H_C present ? True (attendu : True)


  Temoin POSSIBLE verifie : True


ASSERTION PASS : 3 cas (2 IMPOSSIBLE, 1 POSSIBLE) verifies


## 5. Exemples guides -- le temoin en action

Trois exemples guident pas-a-pas l'utilisation du temoin sur des structures differentes des Cas A/B/C (section 3). Chaque exemple combine verification analytique (via la table `D`) et verification operationnelle (via BFS tronque) pour montrer que les deux voies sont d'accord.

### 5.1 Exemple guide -- Un autre cas IMPOSSIBLE

**Demarche** : on choisit une chambre `H` dont la structure est *differente* de l'antipode du Cas A — par exemple, `H` a la **ligne antipode** mais la **colonne identique** a `G`. Alors `d_perm(row_G, row_H) = 6` (antipode du permutaedre) et `d_perm(col_G, col_H) = 0` (meme colonne), donc `d_min = 6 + 0 = 6`. Pour `k_max = 3`, le verdict est `IMPOSSIBLE` puisque `6 > 3`.

**Verification operationnelle** par BFS tronque : on enumere tous les sommets atteignables en `k_max = 3` pas depuis `G`. Si `H` n'y figure pas, le verdict analytique est confirme operationnellement — c'est un **contre-controle** independant du certificat.

L'exemple ci-dessous execute cette verification et imprime les deux verifications (analytique + operationnelle) ; elles doivent etre d'accord. En .NET, la cellule laisse les deux etapes a l'etudiant : le certificat se calcule avec `CertifierImpossibilite(G, H, kMax, D)` et la boule atteignable avec `BfsTronque(G, AdjChambre, kMax)`.

In [6]:
// === Exemple guide 5.1 : un autre cas IMPOSSIBLE (row antipode, col identique) ===
using System;
using System.Collections.Generic;

var G_exemple = new Chambre("1234", "1234");   // G = IDENTITE
var H_exemple = new Chambre("4321", "1234");   // row antipode, col identique
var k_max_exemple = 3;                          // borne a depasser pour un verdict IMPOSSIBLE

// TODO etudiant : etape 1 -- certificat analytique
// cert_exemple = CertifierImpossibilite(G_exemple, H_exemple, k_max_exemple, D)
var cert_exemple = default(Certificat);

// TODO etudiant : etape 2 -- verification operationnelle (BFS tronque, independante de la table)
// atteints_exemple = BfsTronque(G_exemple, AdjChambre, k_max_exemple)
var atteints_exemple = default(HashSet<Chambre>);

if (cert_exemple is null || atteints_exemple is null)
{
    Console.WriteLine("Exemple guide 5.1 a completer");
    Console.WriteLine("// Etape 1 : calculer cert_exemple = CertifierImpossibilite(G_exemple, H_exemple, k_max_exemple, D)");
    Console.WriteLine("// Etape 2 : calculer atteints_exemple = BfsTronque(G_exemple, AdjChambre, k_max_exemple)");
    Console.WriteLine("// Etape 3 : afficher le verdict analytique, le nombre de sommets atteints, la presence de H_exemple");
    Console.WriteLine("//          et dire si les deux verifications sont d'accord");
}
else
{
    bool h_present = atteints_exemple.Contains(H_exemple);
    Console.WriteLine($"Certificat analytique : {cert_exemple.Verdict} - {cert_exemple.Preuve}");
    Console.WriteLine($"BFS tronque : {atteints_exemple.Count} sommets a <= {k_max_exemple} pas");
    Console.WriteLine($"H present dans le voisinage ? {h_present} (attendu : False)");
    Console.WriteLine($"Verifications d'accord : {cert_exemple.Verdict == "IMPOSSIBLE" && !h_present}");
}

Exemple guide 5.1 a completer


// Etape 1 : calculer cert_exemple = CertifierImpossibilite(G_exemple, H_exemple, k_max_exemple, D)


// Etape 2 : calculer atteints_exemple = BfsTronque(G_exemple, AdjChambre, k_max_exemple)


// Etape 3 : afficher le verdict analytique, le nombre de sommets atteints, la presence de H_exemple


//          et dire si les deux verifications sont d'accord


### 5.2 Exemple guide -- Distance d'un antipode

**Demarche** : la distance entre deux sommets `G` et `H` est le plus petit `k` tel que `H` appartient aux sommets a distance `<= k` de `G`. Pour l'antipode `(IDENTITE, RENVERS)`, le certificat analytique donne `d_min = 6 + 6 = 12` — c'est le diametre du graphe produit. Le plus petit `k_max` qui rend `POSSIBLE` est donc `k_max = 12` (a `k_max < 12`, IMPOSSIBLE est certain ; a `k_max >= 12`, POSSIBLE est garanti par concatenation des deux chemins optimaux).

L'exemple ci-dessous balaie les `k` de 0 a 12 et identifie le seuil de bascule. En .NET, la boucle est laissee a l'etudiant : le seuil se lit sur le premier `k` tel que `CertifierImpossibilite(IDENTITE, RENVERS, k, D).Verdict` vaut `POSSIBLE`.

In [7]:
// === Exemple guide 5.2 : distance de l'antipode (IDENTITE, RENVERS) ===
using System;

// TODO etudiant : etape 1 -- balayer k de 0 a 12 et appeler
//                 CertifierImpossibilite(IDENTITE, RENVERS, k, D)
// TODO etudiant : etape 2 -- relever le premier k tel que le verdict est POSSIBLE (seuil de bascule)
// TODO etudiant : etape 3 -- relever la distance de l'antipode (valeur d_min du certificat correspondant)
int seuil = -1;
int d_min_antipode = -1;

if (seuil < 0)
{
    Console.WriteLine("Exemple guide 5.2 a completer");
    Console.WriteLine("// Etape 1 : boucle for (int k = 0; k <= 12; k++) sur CertifierImpossibilite(IDENTITE, RENVERS, k, D)");
    Console.WriteLine("// Etape 2 : au premier verdict POSSIBLE, noter le k dans seuil");
    Console.WriteLine("// Etape 3 : noter la valeur d_min correspondante dans d_min_antipode, puis afficher les deux");
}
else
{
    Console.WriteLine($"Bascule IMPOSSIBLE -> POSSIBLE a k={seuil} (d_min = {d_min_antipode})");
    Console.WriteLine($"Seuil exact : {seuil}, distance de l'antipode = {d_min_antipode}");
}

Exemple guide 5.2 a completer


// Etape 1 : boucle for (int k = 0; k <= 12; k++) sur CertifierImpossibilite(IDENTITE, RENVERS, k, D)


// Etape 2 : au premier verdict POSSIBLE, noter le k dans seuil


// Etape 3 : noter la valeur d_min correspondante dans d_min_antipode, puis afficher les deux


### 5.3 Exemple guide -- Paires `d_row=0, d_col=6`

**Demarche** : `d_row = 0` signifie que la ligne de `G` est identique a la ligne de `H` ; `d_col = 6` signifie que la colonne de `G` est l'antipode de la colonne de `H`. Ce sont donc les paires ou seul le cote colonne doit etre parcoure dans toute sa longueur.

Pour les trouver : on balaie toutes les chambres et on garde celles ou la ligne est inchangee et la colonne est antipode (par convention, l'antipode est `(4, 3, 2, 1)` a droite comme a gauche). Le balayage donne plusieurs solutions ; l'exemple en garde 3 et imprime leur verdict aux seuils `k_max=5` (IMPOSSIBLE) et `k_max=6` (frontiere, POSSIBLE).

En .NET, une chambre est un `record Chambre(string Row, string Col)` et la table `D` est un `Dictionary<(string, string), int>` : le filtrage se lit donc `D[(G.Row, H.Row)]` et `D[(G.Col, H.Col)]`.

In [8]:
// === Exemple guide 5.3 : trois paires d_row=0, d_col=6 ===
using System;
using System.Collections.Generic;

// Collecte des paires (G, H) telles que d_row = 0 et d_col = 6.
var candidates_43 = new List<(Chambre G, Chambre H)>();

// TODO etudiant : etape 1 -- balayer les chambres G puis les chambres H et garder les paires
//   pour lesquelles D[(G.Row, H.Row)] == 0 et D[(G.Col, H.Col)] == 6
// TODO etudiant : etape 2 -- s'arreter des les 3 premieres paires trouvees
// TODO etudiant : etape 3 -- pour chaque paire, afficher le verdict aux seuils 5 puis 6
//   (CertifierImpossibilite(G, H, 5, D) puis CertifierImpossibilite(G, H, 6, D))

if (candidates_43.Count == 0)
{
    Console.WriteLine("Exemple guide 5.3 a completer");
    Console.WriteLine("// Etape 1 : double boucle sur les chambres, filtrer sur D[(G.Row, H.Row)] == 0 et D[(G.Col, H.Col)] == 6");
    Console.WriteLine("// Etape 2 : conserver au plus 3 paires");
    Console.WriteLine("// Etape 3 : afficher les verdicts aux seuils 5 et 6 pour chaque paire");
}
else
{
    int i = 1;
    foreach (var (G, H) in candidates_43)
    {
        var c5 = CertifierImpossibilite(G, H, 5, D);
        var c6 = CertifierImpossibilite(G, H, 6, D);
        Console.WriteLine($"Paire {i} : k_max=5 -> {c5.Verdict}, k_max=6 -> {c6.Verdict}");
        i++;
    }
}

Exemple guide 5.3 a completer


// Etape 1 : double boucle sur les chambres, filtrer sur D[(G.Row, H.Row)] == 0 et D[(G.Col, H.Col)] == 6


// Etape 2 : conserver au plus 3 paires


// Etape 3 : afficher les verdicts aux seuils 5 et 6 pour chaque paire


## 6. Inventaire -- combien de triplets sont IMPOSSIBLE ?

Echantillon aleatoire de 100 chambres representatives (seed=42 pour la reproductibilite), balayage des `k_max ∈ {0..11}`. La proportion d'IMPOSSIBLE **decroît quand `k_max` augmente** (plus la borne est large, plus il est facile de tomber dans la zone POSSIBLE ; plus elle est serree, plus l'impossibilite est facile a prouver), puis s'annule au-dela du diametre (`k_max >= 12` : tout couple devient POSSIBLE).

La sortie de la cellule code plus loin trace les valeurs reelles sur le sous-echantillon :

| `k_max` | IMPOSSIBLE / Total | % |
|---:|---:|---:|
| 0 | 57 500 / 57 500 | 100,0 % |
| 1 | 56 900 / 57 500 | 99,0 % |
| 2 | 55 000 / 57 500 | 95,7 % |
| 3 | 50 800 / 57 500 | 88,3 % |
| 4 | 43 700 / 57 500 | 76,0 % |
| 5 | 34 100 / 57 500 | 59,3 % |

Ces valeurs sont celles qu'imprime la cellule code qui suit immediatement, reexecutees sur le noyau .NET : les douze compteurs et les pourcentages coincident terme a terme avec ceux de la version Python du carnet (meme echantillon de 100 chambres, memes seuils).

En .NET, l'echantillon de 100 chambres est fige par une liste d'indices (le tirage `random.sample` de CPython n'est pas reproductible par `System.Random`, dont l'algorithme differe), et les pourcentages sont affiches avec `CultureInfo.InvariantCulture` pour garder le point decimal de la source.

In [9]:
// === Section 6.1 : inventaire ===
using System;
using System.Globalization;
using System.Linq;

// Echantillon de 100 chambres representatives : l'echantillon de la source Python
// (random.sample(chambres, 100), seed=42) est fige ici sous forme d'une liste d'indices
// (0..575), car System.Random n'implemente pas le generateur de Mersenne Twister de
// CPython. La procedure de generation du litteral (tirage CPython seed=42 sur l'ordre
// des chambres, puis transcription des indices) est consignee dans l'attestation de
// parite ; la parite numerique est verifiee ici par les compteurs affiches, identiques
// a ceux de la source pour les douze seuils k_max = 0..11.
int[] echantillon_idx = {
    114, 25, 281, 250, 228, 142, 104, 558, 89, 432,
    32, 30, 95, 223, 238, 517, 27, 203, 429, 225,
    459, 284, 6, 163, 566, 348, 554, 159, 220, 344,
    569, 94, 389, 99, 367, 352, 270, 44, 470, 127,
    387, 80, 300, 370, 196, 71, 46, 233, 296, 81,
    561, 103, 543, 549, 464, 373, 166, 379, 363, 214,
    273, 73, 175, 572, 167, 236, 194, 138, 506, 473,
    327, 540, 285, 112, 350, 519, 431, 393, 397, 28,
    117, 420, 16, 412, 161, 205, 137, 33, 108, 467,
    482, 290, 448, 541, 491, 487, 335, 255, 202, 452
};
var echantillon = echantillon_idx.Select(i => chambres[i]).ToList();
Console.WriteLine($"Echantillon : {echantillon.Count} chambres, {echantillon.Distinct().Count()} distinctes");

// Comptage par la table D (voie analytique).
var resultats_par_k = new Dictionary<int, (int Imp, int Tot)>();
for (int k_max = 0; k_max < 12; k_max++)
{
    int compteur_imp = 0;
    int compteur_tot = 0;
    foreach (var G in echantillon)
    {
        foreach (var H in chambres)
        {
            if (G.Equals(H))
            {
                continue;
            }
            int d_min = D[(G.Row, H.Row)] + D[(G.Col, H.Col)];
            compteur_tot++;
            if (d_min > k_max)
            {
                compteur_imp++;
            }
        }
    }
    resultats_par_k[k_max] = (compteur_imp, compteur_tot);
}

Console.WriteLine("k_max | IMPOSSIBLE / Total | %");
Console.WriteLine(new string('-', 50));
for (int k_max = 0; k_max < 12; k_max++)
{
    var (imp, tot) = resultats_par_k[k_max];
    string pct = (100.0 * imp / tot).ToString("0.0", CultureInfo.InvariantCulture);
    Console.WriteLine($"  {k_max,2}  | {imp,6} / {tot,6} | {pct,5}%");
}

// Controle croise : meme comptage par BFS depuis chaque G (voie operationnelle).
var resultats_par_k_bfs = new Dictionary<int, int>();
foreach (var G in echantillon)
{
    var distG = BfsChambre(G, AdjChambre);
    for (int k_max = 0; k_max < 12; k_max++)
    {
        int imp_bfs = 0;
        foreach (var H in chambres)
        {
            if (H.Equals(G))
            {
                continue;
            }
            if (distG[H] > k_max)
            {
                imp_bfs++;
            }
        }
        resultats_par_k_bfs[k_max] = resultats_par_k_bfs.TryGetValue(k_max, out int cumul) ? cumul + imp_bfs : imp_bfs;
    }
}
int ecarts_croises = 0;
for (int k_max = 0; k_max < 12; k_max++)
{
    if (resultats_par_k_bfs[k_max] != resultats_par_k[k_max].Imp)
    {
        ecarts_croises++;
    }
}
Console.WriteLine();
Console.WriteLine($"Controle croise par BFS (une source par echantillon) : {ecarts_croises} ecart(s) sur 12 seuils -> {(ecarts_croises == 0 ? "CONFORME" : "ECART")}");

int imp5 = resultats_par_k[5].Imp;
int tot5 = resultats_par_k[5].Tot;
Console.WriteLine();
Console.WriteLine($"A k_max=5 : {(100.0 * imp5 / tot5).ToString("0.0", CultureInfo.InvariantCulture)}% des paires sont IMPOSSIBLE -- le temoin tranche souvent");

Echantillon : 100 chambres, 100 distinctes


k_max | IMPOSSIBLE / Total | %


--------------------------------------------------


   0  |  57500 /  57500 | 100.0%


   1  |  56900 /  57500 |  99.0%


   2  |  55000 /  57500 |  95.7%


... (7 outputs left out)

  10  |    700 /  57500 |   1.2%


  11  |    100 /  57500 |   0.2%


Controle croise par BFS (une source par echantillon) : 0 ecart(s) sur 12 seuils -> CONFORME


A k_max=5 : 59.3% des paires sont IMPOSSIBLE -- le temoin tranche souvent


## 7. Frontiere et bornes

**Certifie** : pour `k_max < 12`, IMPOSSIBLE est certain des que `d_min > k_max`.

**Non certifie** : un cas POSSIBLE garantit l'**existence** d'un chemin, pas un chemin explicite. La construction explicite est deléguée au constructeur de GT-20 (point 1 du critere #12205 §5) qui produit le chemin par concatenation des chemins optimaux dans chaque facteur.

**Au-dela de `k_max = 12`** (le diametre du produit), tout est POSSIBLE par definition : le chemin existe toujours en au plus 12 pas.

In [10]:
// === Section 7.1 : resume final ===
using System;
using System.Globalization;
using System.Linq;

// Ecarts de la structure produit (recalcules, jamais repris en dur).
int conformes = chambres.Count(c => dist_chambre[c] == dist_perm[IDENTITE.Row][c.Row] + dist_perm[IDENTITE.Col][c.Col]);
int ecarts_structure = chambres.Count - conformes;

// Symetrie et triangularite de la table 24x24 (recalculees).
int asymetries = 0;
foreach (var o1 in stricts)
{
    foreach (var o2 in stricts)
    {
        if (D[(o1, o2)] != D[(o2, o1)])
        {
            asymetries++;
        }
    }
}
int violations = 0;
foreach (var o1 in stricts)
{
    foreach (var o2 in stricts)
    {
        foreach (var o3 in stricts)
        {
            if (D[(o1, o3)] > D[(o1, o2)] + D[(o2, o3)])
            {
                violations++;
            }
        }
    }
}

Console.WriteLine("RESUME -- Le point 4 du critere #12205 est TENU sur Robinson-Goforth");
Console.WriteLine();
Console.WriteLine("Verdict par cas :");
Console.WriteLine($"  CAS A : {cas_A.Verdict}");
Console.WriteLine($"  CAS B : {cas_B.Verdict}");
Console.WriteLine($"  CAS C : {cas_C.Verdict}");
Console.WriteLine();
Console.WriteLine("Verification par BFS tronque :");
Console.WriteLine($"  CAS A IMPOSSIBLE verifie : {verif_A}");
Console.WriteLine($"  CAS B IMPOSSIBLE verifie : {verif_B}");
Console.WriteLine($"  CAS C POSSIBLE verifie   : {verif_C}");
Console.WriteLine();
Console.WriteLine($"Structure produit : {conformes}/{chambres.Count} chambres, {ecarts_structure} ecart");
Console.WriteLine($"Table 24x24 : {asymetries} asymetrie(s), {violations} violation(s) de l'inegalite triangulaire");
Console.WriteLine($"A k_max=5 : {(100.0 * imp5 / tot5).ToString("0.0", CultureInfo.InvariantCulture)}% des paires sont IMPOSSIBLE");
Console.WriteLine();
Console.WriteLine($"Synthese : {(ecarts_structure == 0 && asymetries == 0 && violations == 0 && verif_A && verif_B && verif_C ? "CONFORME" : "ECART")}");

RESUME -- Le point 4 du critere #12205 est TENU sur Robinson-Goforth


Verdict par cas :


  CAS A : IMPOSSIBLE


  CAS B : IMPOSSIBLE


  CAS C : POSSIBLE


Verification par BFS tronque :


... (3 outputs left out)

Structure produit : 576/576 chambres, 0 ecart


Table 24x24 : 0 asymetrie(s), 0 violation(s) de l'inegalite triangulaire


A k_max=5 : 59.3% des paires sont IMPOSSIBLE


Synthese : CONFORME


## 8. Exercices

### Exercice A -- Un cas IMPOSSIBLE non antipode

Trouver un couple `(G, H)` et un `k_max` tels que le verdict est `IMPOSSIBLE`, avec une **structure differente** des cas fondateurs suivants :

- **Cas A** : antipode integral (les deux cotes a distance 6) -- exhibe en cellule 8 ;
- **Cas B** (`d_row = 0, d_col = 4`) : un seul cote avec une distance **non antipode** (4 < 6) ; aussi exhibe en cellule 8 ;
- **Exemple 5.1** (`H_exemple` = `((4, 3, 2, 1), (1, 2, 3, 4))`) : un seul cote **antipode** (`d_perm(row_G, row_H) = 6`, `d_perm(col_G, col_H) = 0`) -- l'exemple guide de la section 5.

L'enonce de l'exercice exclut donc l'antipode integral (Cas A) **et** les mono-cotes antipodes (exemple 5.1) **et** les mono-cotes non antipodes (Cas B). Reste ce qui n'appartient a aucune de ces trois familles : par exemple, les deux cotes a des distances intermediaires (3 et 4, par exemple) pour un `k_max = 5`. Verifier par BFS tronque que le verdict certifie est bien `IMPOSSIBLE`.

En .NET, une chambre se construit avec `new Chambre("1234", "3412")` (ligne, colonne), le certificat se calcule avec `CertifierImpossibilite(G, H, kMax, D)` et le contre-controle avec `BfsTronque(G, AdjChambre, kMax)`.

In [11]:
// Exercice A : cas IMPOSSIBLE non antipode
// A l'etudiant : choisir G, H, k_max de sorte que d_row + d_col > k_max
// avec une structure differente des Cas A/B/5.1.
using System;
using System.Collections.Generic;

Chambre G_A = default;         // TODO etudiant
Chambre H_A = default;         // TODO etudiant
int k_max_A = -1;               // TODO etudiant
Certificat resultat_A = default; // TODO etudiant

// Verification par l'etudiant :
// 1. resultat_A = CertifierImpossibilite(G_A, H_A, k_max_A, D)
// 2. var atteints = BfsTronque(G_A, AdjChambre, k_max_A)
// 3. assert resultat_A.Verdict == "IMPOSSIBLE" && !atteins.Contains(H_A)

if (G_A is null || H_A is null || k_max_A < 0)
{
    Console.WriteLine("Exercice A a completer");
    Console.WriteLine("// Etape 1 : choisir G_A et H_A (deux chambres) puis k_max_A");
    Console.WriteLine("// Etape 2 : resultat_A = CertifierImpossibilite(G_A, H_A, k_max_A, D)");
    Console.WriteLine("// Etape 3 : atteints = BfsTronque(G_A, AdjChambre, k_max_A), puis verifier le verdict et l'absence de H_A");
}
else
{
    var atteints = BfsTronque(G_A, AdjChambre, k_max_A);
    Console.WriteLine($"Exercice A : {resultat_A.Verdict} - {resultat_A.Preuve}");
    Console.WriteLine($"  sommets a <= {k_max_A} pas : {atteints.Count}");
    Console.WriteLine($"  H_A absent de la boule ? {!atteints.Contains(H_A)} (attendu : True)");
}

Exercice A a completer


// Etape 1 : choisir G_A et H_A (deux chambres) puis k_max_A


// Etape 2 : resultat_A = CertifierImpossibilite(G_A, H_A, k_max_A, D)


// Etape 3 : atteints = BfsTronque(G_A, AdjChambre, k_max_A), puis verifier le verdict et l'absence de H_A


### Exercice B -- Structure produit pour un G arbitraire

Verifier que la structure produit reste valide pour un `G` de depart qui n'est **pas** `IDENTITE` — par exemple `G = ((2, 1, 4, 3), (3, 4, 1, 2))`. La structure produit est-elle toujours `d_chambre(G, H) = d_perm(row_G, row_H) + d_perm(col_G, col_H)`, ou faut-il la reformuler ?

En .NET, `G_B` se construit avec `new Chambre("2143", "3412")`, le BFS depuis `G_B` avec `BfsChambre(G_B, AdjChambre)` et la somme des deux cotes avec `D[(G_B.Row, H.Row)] + D[(G_B.Col, H.Col)]`.

In [12]:
// Exercice B : verifier la structure produit pour un G != IDENTITE
using System;
using System.Collections.Generic;
using System.Linq;

var G_B = new Chambre("2143", "3412");  // un G arbitraire (non IDENTITE)

// A l'etudiant :
// 1. Calculer dist_chambre_G = BfsChambre(G_B, AdjChambre)
// 2. Pour chaque H in chambres, comparer dist_chambre_G[H] a
//    D[(G_B.Row, H.Row)] + D[(G_B.Col, H.Col)]
// 3. Conclure : la structure produit tient-elle ? Pourquoi ?
int[] ecarts_B = default;            // TODO etudiant
string resultat_B = default;          // TODO etudiant

if (ecarts_B is null || resultat_B is null)
{
    Console.WriteLine("Exercice B a completer");
    Console.WriteLine("// Etape 1 : dist_chambre_G = BfsChambre(G_B, AdjChambre)");
    Console.WriteLine("// Etape 2 : pour chaque H de chambres, comparer dist_chambre_G[H] a D[(G_B.Row, H.Row)] + D[(G_B.Col, H.Col)] et retenir les ecarts dans ecarts_B");
    Console.WriteLine("// Etape 3 : conclure dans resultat_B (la structure produit tient-elle ? pourquoi ?)");
}
else
{
    Console.WriteLine($"Exercice B : {ecarts_B.Length} ecart(s) sur {chambres.Count} chambres");
    foreach (int i in ecarts_B.Take(10))
    {
        Console.WriteLine($"  ECART : {chambres[i]}");
    }
    Console.WriteLine($"Conclusion : {resultat_B}");
}

Exercice B a completer


// Etape 1 : dist_chambre_G = BfsChambre(G_B, AdjChambre)


// Etape 2 : pour chaque H de chambres, comparer dist_chambre_G[H] a D[(G_B.Row, H.Row)] + D[(G_B.Col, H.Col)] et retenir les ecarts dans ecarts_B


// Etape 3 : conclure dans resultat_B (la structure produit tient-elle ? pourquoi ?)


### Exercice C -- Densite d'IMPOSSIBLE a k_max=7 par echantillonnage stratifie

La section 6 a montre une densite a `k_max=5` sur 100 chambres aleatoires. Estimer la densite a `k_max=7` en utilisant un echantillon **stratifie** : le scaffold ci-dessous partitionne le permutaedre en **7 strates** selon la distance a `IDENTITE` (0 a 6) et tire **4 chambres dans chaque strate**. Completer le comptage de la proportion d'IMPOSSIBLE, puis comparer au chiffre donne par la section 6 (qui utilise un echantillon aleatoire non stratifie).

En .NET, les 7 strates et les 20 ordres tires sont figes en litteral (le tirage `random.sample` avec `seed=2025` de CPython n'est pas reproductible par `System.Random`) ; les ordres tires sont des lignes, la colonne de chaque chambre reste a determiner par l'etudiant.

In [13]:
// Exercice C : densite IMPOSSIBLE a k_max=7 par echantillonnage stratifie
using System;
using System.Collections.Generic;
using System.Linq;

// Stratification par distance a IDENTITE dans le permutaedre.
// Les 7 strates sont figees en litteral : le tirage Python random.sample avec seed=2025
// n'est pas reproductible par System.Random (algorithme different de Mersenne Twister).
// La procedure de generation du litteral (tirage CPython seed=2025, 4 ordres par strate)
// est consignee dans l'attestation de parite. Aucun engagement de parite numerique n'est
// porte ici : dans la version Python du carnet, cet exercice est non corrige et sans sortie.
var strates = new Dictionary<int, List<string>>
{
    [0] = new List<string> { "1234" },
    [1] = new List<string> { "1243", "1324", "2134" },
    [2] = new List<string> { "1342", "1423", "2143", "2314", "3124" },
    [3] = new List<string> { "1432", "2341", "2413", "3142", "3214", "4123" },
    [4] = new List<string> { "2431", "3241", "3412", "4132", "4213" },
    [5] = new List<string> { "3421", "4231", "4312" },
    [6] = new List<string> { "4321" },
};

// Echantillon stratifie : 4 ordres tires par strate (20 elements, tous distincts).
var echantillon_stratifie = new List<string>
{
    "1234",
    "2134", "1324", "1243",
    "3124", "1342", "1423", "2143",
    "3214", "2341", "1432", "4123",
    "4132", "2431", "4213", "3412",
    "3421", "4231", "4312",
    "4321",
};

Console.WriteLine($"Strates : {strates.Count}, effectifs : {string.Join(", ", strates.OrderBy(kv => kv.Key).Select(kv => $"d={kv.Key}:{kv.Value.Count}"))}");
Console.WriteLine($"Echantillon stratifie : {echantillon_stratifie.Count} elements, {echantillon_stratifie.Distinct().Count()} distincts");

// TODO etudiant : chaque ordre tire donne une chambre -- choisir la colonne (par exemple identique a l'identite)
// TODO etudiant : 1. Pour chaque G dans echantillon_stratifie, pour chaque H dans chambres,
// TODO etudiant :    compter les IMPOSSIBLE a k_max=7 (en evitant G == H)
// TODO etudiant : 2. Comparer au chiffre aleatoire de la section 6 (k_max=5)
int[] resultat_C = default;  // TODO etudiant

if (resultat_C is null)
{
    Console.WriteLine("Exercice C a completer");
    Console.WriteLine("// Etape 1 : construire les chambres de l'echantillon stratifie (ligne = ordre tire, colonne a choisir)");
    Console.WriteLine("// Etape 2 : compter les IMPOSSIBLE a k_max=7 via D[(G.Row, H.Row)] + D[(G.Col, H.Col)] > 7");
    Console.WriteLine("// Etape 3 : comparer la proportion obtenue au chiffre de la section 6 et consigner le resultat dans resultat_C");
}
else
{
    Console.WriteLine($"Exercice C : {resultat_C.Length} mesure(s) reponse(s)");
    foreach (int v in resultat_C)
    {
        Console.WriteLine($"  {v}");
    }
}

Strates : 7, effectifs : d=0:1, d=1:3, d=2:5, d=3:6, d=4:5, d=5:3, d=6:1


Echantillon stratifie : 20 elements, 20 distincts


Exercice C a completer


// Etape 1 : construire les chambres de l'echantillon stratifie (ligne = ordre tire, colonne a choisir)


// Etape 2 : compter les IMPOSSIBLE a k_max=7 via D[(G.Row, H.Row)] + D[(G.Col, H.Col)] > 7


// Etape 3 : comparer la proportion obtenue au chiffre de la section 6 et consigner le resultat dans resultat_C


## Conclusion -- le point 4 sur Robinson-Goforth est tenu

**Acceptance #12205 §5** :

1. Substrat dont le verificateur existe sur main : GT-20 porte les points 1-3. **OUI**.
2. Generateur != verificateur : `CertifierImpossibilite` (O(1)) != BFS exhaustif (O(V + E) sur la boule visitée, ici O(576 + 3 456) par palier). **OUI**.
3. Temoin certifie : verdict IMPOSSIBLE porte certificat analytique (d_row, d_col, k_max). **OUI**.
4. Cas sans solution = temoin d'impossibilite, pas silence : section 4 verifie par BFS tronque. Section 6 : plus de la moitie des paires a k_max=5 sont IMPOSSIBLE. **OUI**.

**Structure du notebook** :
- Sections 1-2 : preuve analytique de la structure produit (borne inf + borne sup) ;
- Section 3 : 3 cas fondateurs A/B/C ;
- Section 4 : verification operationnelle par BFS tronque ;
- Section 5 : 3 exemples guides integres a la narration (5.1, 5.2, 5.3) ;
- Sections 6-7 : inventaire, frontiere et bornes ;
- Section 8 : 3 exercices non corriges (A, B, C) avec stubs `// TODO etudiant`.

**Substrats independants portant les 4 points** :
- Life/Conway (livree par la serie Life)
- AMD (`impossibility_strict_payments`)
- Robinson-Goforth (ce notebook)
- Tweety-5d (`afB_no_stable`)

Reference croisee : #12205.

## À retenir

- **Un cas sans solution est un *témoin d'impossibilité*, pas un silence.** Le critère #12205 §5 est tenu sur Robinson-Goforth : chaque verdict `IMPOSSIBLE` porte un **certificat analytique** `(d_row, d_col, k_max)` produit par `CertifierImpossibilite` (O(1)), distinct du BFS exhaustif qui le *vérifie*.
- **La structure produit est le levier analytique** : `d_chambre(G, H) = d_perm(row_G, row_H) + d_perm(col_G, col_H)`. Elle réduit un problème de 576 sommets à deux problèmes de 24, et rend le certificat trivial à produire.
- **L'impossibilité est fréquente, pas marginale** : à `k_max = 5`, plus de la moitié des paires de chambres sont `IMPOSSIBLE` (mesuré `59.3%`). Le témoin n'est pas un artefact du choix des cas.

## Vérifiez votre compréhension

1. Pourquoi le générateur (`CertifierImpossibilite`, O(1)) et le vérificateur (BFS exhaustif) doivent-ils être *différents* ? Que certifie chacun ?
2. En quoi la structure produit `d_row + d_col` rend-elle le certificat d'impossibilité trivial à produire ?
3. Pourquoi le chiffre `59.3%` de paires impossibles à `k_max = 5` renforce-t-il la valeur du témoin plutôt qu'elle ne la limite ?

## Pour aller plus loin

- **Les substrats indépendants qui portent les 4 points du critère** : Life/Conway, AMD (`impossibility_strict_payments`), Tweety-5d (`afB_no_stable`).
- **L'échantillonnage stratifié** (exercice C) : estimer une densité sur un permutaèdre en partitionnant par distance à l'identité.
- Le lien avec les témoins d'impossibilité en théorie des jeux bayésiens.